Train Location Models — Live Production Data
Trains weekly (LightGBM) and monthly (Prophet) models per location, using hybrid model-vs-baseline selection, reconciled to the overall forecast.

**Input**: gold/erp/battery/phase4_location_weekly_live.parquet, phase4_location_monthly_live.parquet
**Output**: gold/live/forecasts/battery/active + history (JSON + Excel) per location

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold, append_json_history, save_history_as_excel
import pandas as pd
import json
import io
import datetime
import lightgbm as lgb
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

FORECAST_BASE = "live/battery"

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

In [0]:
location_reference = read_gold(blob_service, f"{FORECAST_BASE}/data/reference/location_reference_live.parquet")
location_ref_dict = location_reference.set_index("location_code").to_dict(orient="index")

print(location_reference)

Load weekly gold

In [0]:
gold_location_weekly = read_gold(blob_service, f"{FORECAST_BASE}/data/phase4_location_weekly_live.parquet")
gold_location_weekly["week_start"] = pd.to_datetime(gold_location_weekly["week_start"])
gold_location_weekly["location_code"] = gold_location_weekly["location_code"].astype("category")

locations = gold_location_weekly["location_code"].cat.categories.tolist()
print(locations)

Weekly: train/test split, train, evaluate per location

In [0]:
feature_cols_location = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "location_code"]
target_col = "total_units_sold"

model_data_location = gold_location_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_location = model_data_location.sort_values("week_start")

split_idx = int(len(model_data_location) * 0.8)
train_location = model_data_location.iloc[:split_idx]
test_location = model_data_location.iloc[split_idx:]

X_train_l, y_train_l = train_location[feature_cols_location], train_location[target_col]
X_test_l, y_test_l = test_location[feature_cols_location], test_location[target_col]

model_location = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
model_location.fit(X_train_l, y_train_l, categorical_feature=["location_code"])

preds_l = model_location.predict(X_test_l)
test_location_results = test_location.copy()
test_location_results["prediction"] = preds_l

location_weekly_method = {}
for loc in test_location_results["location_code"].unique():
    subset = test_location_results[test_location_results["location_code"] == loc]
    model_wape = wape(subset["total_units_sold"], subset["prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
    location_weekly_method[loc] = "model" if model_wape < baseline_wape else "baseline"
    print(f"{loc:15s} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {location_weekly_method[loc]}")

Weekly: retrain on all data, predict next week per location

In [0]:
final_location_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_l = model_data_location[feature_cols_location]
y_all_l = model_data_location[target_col]
final_location_model.fit(X_all_l, y_all_l, categorical_feature=["location_code"])

last_week_start = gold_location_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

location_weekly_predictions = {}
for loc in locations:
    loc_hist = gold_location_weekly[gold_location_weekly["location_code"] == loc]

    if location_weekly_method.get(loc) == "model":
        lag_val = loc_hist[loc_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": loc_hist["total_units_sold"].tail(4).mean(),
            "location_code": loc,
        }])
        row["location_code"] = row["location_code"].astype("category")
        pred = final_location_model.predict(row[feature_cols_location])[0]
    else:
        pred = loc_hist["total_units_sold"].tail(4).mean()

    location_weekly_predictions[loc] = max(pred, 0)
    print(f"{loc:15s}: {location_weekly_predictions[loc]:.0f}")

Reconcile weekly to overall active forecast

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly_overall = json.loads(stream)

overall_target = next(
    (w["predicted_units"] for w in active_weekly_overall if w["week_start"] == next_week_start.strftime("%Y-%m-%d")),
    sum(location_weekly_predictions.values())
)

location_sum = sum(location_weekly_predictions.values())
location_weekly_reconciled = {
    loc: (val / location_sum) * overall_target if location_sum > 0 else 0
    for loc, val in location_weekly_predictions.items()
}

print(f"Overall target: {overall_target}   Location sum (raw): {location_sum:.0f}   Location sum (reconciled): {sum(location_weekly_reconciled.values()):.0f}")
for loc, val in sorted(location_weekly_reconciled.items(), key=lambda x: -x[1]):
    print(f"{loc:15s}: {val:.0f}")

Load monthly gold, per-location Prophet with hybrid selection

In [0]:
gold_location_monthly = read_gold(blob_service, f"{FORECAST_BASE}/data/phase4_location_monthly_live.parquet")
gold_location_monthly["month_start"] = pd.to_datetime(gold_location_monthly["month_start"])

location_monthly_method = {}
location_monthly_forecast = {}

for loc in locations:
    loc_df = gold_location_monthly[gold_location_monthly["location_code"] == loc][["month_start", "total_units_sold"]]
    loc_df = loc_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(loc_df) < 15 or loc_df["y"].tail(12).sum() == 0:
        method = "baseline"
    else:
        train_l = loc_df.iloc[:-3]
        test_l = loc_df.iloc[-3:]
        try:
            m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
            m_test.fit(train_l)
            future_test = m_test.make_future_dataframe(periods=3, freq="MS")
            forecast_test = m_test.predict(future_test)
            test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

            model_wape = wape(test_l["y"].values, test_preds)
            naive_pred = train_l["y"].tail(3).mean()
            baseline_wape = wape(test_l["y"].values, [naive_pred] * 3)
            method = "model" if model_wape < baseline_wape else "baseline"
            print(f"{loc:15s} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {method}")
        except Exception:
            method = "baseline"
            print(f"{loc:15s} — Prophet failed, using baseline")

    location_monthly_method[loc] = method

    if method == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
        m_final.fit(loc_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
        month_labels = forecast_final.tail(3)["ds"].dt.strftime("%Y-%m-%d").tolist()
    else:
        flat_value = max(loc_df["y"].tail(3).mean(), 0)
        preds = [flat_value] * 3
        last_month = loc_df["ds"].max()
        month_labels = [(last_month + pd.DateOffset(months=i)).strftime("%Y-%m-01") for i in range(1, 4)]

    location_monthly_forecast[loc] = {"months": month_labels, "values": [float(p) for p in preds]}

Reconcile monthly to overall active forecast, per month

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly_overall = json.loads(stream)

overall_by_month = {m["month_start"]: m["predicted_units"] for m in active_monthly_overall}

location_monthly_reconciled = {loc: {"months": [], "values": []} for loc in locations}

sample_months = location_monthly_forecast[locations[0]]["months"]
for i, month_label in enumerate(sample_months):
    month_location_sum = sum(location_monthly_forecast[loc]["values"][i] for loc in locations)
    overall_target = overall_by_month.get(month_label, month_location_sum)

    for loc in locations:
        raw_val = location_monthly_forecast[loc]["values"][i]
        reconciled_val = (raw_val / month_location_sum) * overall_target if month_location_sum > 0 else 0
        location_monthly_reconciled[loc]["months"].append(month_label)
        location_monthly_reconciled[loc]["values"].append(reconciled_val)

for loc, data in location_monthly_reconciled.items():
    print(f"{loc}: {[f'{v:.0f}' for v in data['values']]}")

Save active + history + Excel (weekly)

In [0]:
today_str = datetime.date.today().isoformat()
today = pd.Timestamp(datetime.date.today())

weekly_records = [
    {
        "generated_date": today_str,
        "week_start": next_week_start.strftime("%Y-%m-%d"),
        "week_end": (next_week_start + pd.Timedelta(days=6)).strftime("%Y-%m-%d"),
        "location_code": loc,
        "location_description": location_ref_dict.get(loc, {}).get("location_description", "Unknown"),
        "district_name": location_ref_dict.get(loc, {}).get("district_name", "Others"),
        "province_name": location_ref_dict.get(loc, {}).get("province_name", "Others"),
        "predicted_units": round(float(val))
    }
    for loc, val in location_weekly_reconciled.items()
]

weekly_history = append_json_history(blob_service, weekly_records, f"{FORECAST_BASE}/forecasts/history/location_weekly_forecast_history.json")

weekly_df = pd.DataFrame(weekly_history)
weekly_df["week_end"] = pd.to_datetime(weekly_df["week_end"])
weekly_df["generated_date"] = pd.to_datetime(weekly_df["generated_date"])

active_weekly_loc = weekly_df[weekly_df["week_end"] >= today]
active_weekly_loc = active_weekly_loc.sort_values("generated_date").drop_duplicates(subset=["week_start", "location_code"], keep="last")
active_weekly_loc = active_weekly_loc.sort_values(["week_start", "location_code"])

active_weekly_loc_records = active_weekly_loc.to_dict(orient="records")
for r in active_weekly_loc_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/location_weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_weekly_loc_records, indent=2), overwrite=True)
print(f"Active location weekly: {len(active_weekly_loc_records)} records")

count = save_history_as_excel(blob_service, weekly_history, f"{FORECAST_BASE}/forecasts/history/location_weekly_forecast_history.xlsx")
print(f"Saved location weekly Excel: {count} rows")

Save active + history + Excel (monthly)

In [0]:
monthly_records = []
for loc, data in location_monthly_reconciled.items():
    for month_label, val in zip(data["months"], data["values"]):
        monthly_records.append({
            "generated_date": today_str,
            "month_start": month_label,
            "location_code": loc,
            "location_description": location_ref_dict.get(loc, {}).get("location_description", "Unknown"),
            "district_name": location_ref_dict.get(loc, {}).get("district_name", "Others"),
            "province_name": location_ref_dict.get(loc, {}).get("province_name", "Others"),
            "predicted_units": round(float(val))
        })

monthly_history = append_json_history(blob_service, monthly_records, f"{FORECAST_BASE}/forecasts/history/location_monthly_forecast_history.json")

monthly_df = pd.DataFrame(monthly_history)
monthly_df["month_start"] = pd.to_datetime(monthly_df["month_start"])
monthly_df["generated_date"] = pd.to_datetime(monthly_df["generated_date"])
monthly_df["month_end"] = monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_monthly_loc = monthly_df[monthly_df["month_end"] >= today]
active_monthly_loc = active_monthly_loc.sort_values("generated_date").drop_duplicates(subset=["month_start", "location_code"], keep="last")
active_monthly_loc = active_monthly_loc.sort_values(["month_start", "location_code"])

active_monthly_loc_records = active_monthly_loc.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_monthly_loc_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/location_monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_monthly_loc_records, indent=2), overwrite=True)
print(f"Active location monthly: {len(active_monthly_loc_records)} records")

count = save_history_as_excel(blob_service, monthly_history, f"{FORECAST_BASE}/forecasts/history/location_monthly_forecast_history.xlsx")
print(f"Saved location monthly Excel: {count} rows")

In [0]:
weekly_with_geo = pd.DataFrame(weekly_records)
monthly_with_geo = pd.DataFrame(monthly_records)

print("By district (weekly):")
print(weekly_with_geo.groupby("district_name")["predicted_units"].sum().sort_values(ascending=False))

print("\nBy province (weekly):")
print(weekly_with_geo.groupby("province_name")["predicted_units"].sum().sort_values(ascending=False))

Save district - weekly

In [0]:
district_weekly_records = weekly_with_geo.groupby(["district_name", "week_start", "week_end"])["predicted_units"].sum().reset_index()
district_weekly_records["generated_date"] = today_str
district_weekly_records = district_weekly_records.rename(columns={"district_name": "district"}).to_dict(orient="records")

district_weekly_history = append_json_history(blob_service, district_weekly_records, f"{FORECAST_BASE}/forecasts/history/district_weekly_forecast_history.json")

district_weekly_df = pd.DataFrame(district_weekly_history)
district_weekly_df["week_end"] = pd.to_datetime(district_weekly_df["week_end"])
district_weekly_df["generated_date"] = pd.to_datetime(district_weekly_df["generated_date"])

active_district_weekly = district_weekly_df[district_weekly_df["week_end"] >= today]
active_district_weekly = active_district_weekly.sort_values("generated_date").drop_duplicates(subset=["week_start", "district"], keep="last")
active_district_weekly = active_district_weekly.sort_values(["week_start", "district"])

active_district_weekly_records = active_district_weekly.to_dict(orient="records")
for r in active_district_weekly_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/district_weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_district_weekly_records, indent=2), overwrite=True)
print(f"Active district weekly: {len(active_district_weekly_records)} records")

count = save_history_as_excel(blob_service, district_weekly_history, f"{FORECAST_BASE}/forecasts/history/district_weekly_forecast_history.xlsx")
print(f"Saved district weekly Excel: {count} rows")

Save district - monthly

In [0]:
district_monthly_records = monthly_with_geo.groupby(["district_name", "month_start"])["predicted_units"].sum().reset_index()
district_monthly_records["generated_date"] = today_str
district_monthly_records = district_monthly_records.rename(columns={"district_name": "district"}).to_dict(orient="records")

district_monthly_history = append_json_history(blob_service, district_monthly_records, f"{FORECAST_BASE}/forecasts/history/district_monthly_forecast_history.json")

district_monthly_df = pd.DataFrame(district_monthly_history)
district_monthly_df["month_start"] = pd.to_datetime(district_monthly_df["month_start"])
district_monthly_df["generated_date"] = pd.to_datetime(district_monthly_df["generated_date"])
district_monthly_df["month_end"] = district_monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_district_monthly = district_monthly_df[district_monthly_df["month_end"] >= today]
active_district_monthly = active_district_monthly.sort_values("generated_date").drop_duplicates(subset=["month_start", "district"], keep="last")
active_district_monthly = active_district_monthly.sort_values(["month_start", "district"])

active_district_monthly_records = active_district_monthly.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_district_monthly_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/district_monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_district_monthly_records, indent=2), overwrite=True)
print(f"Active district monthly: {len(active_district_monthly_records)} records")

count = save_history_as_excel(blob_service, district_monthly_history, f"{FORECAST_BASE}/forecasts/history/district_monthly_forecast_history.xlsx")
print(f"Saved district monthly Excel: {count} rows")

Save province - weekly

In [0]:
province_weekly_records = weekly_with_geo.groupby(["province_name", "week_start", "week_end"])["predicted_units"].sum().reset_index()
province_weekly_records["generated_date"] = today_str
province_weekly_records = province_weekly_records.rename(columns={"province_name": "province"}).to_dict(orient="records")

province_weekly_history = append_json_history(blob_service, province_weekly_records, f"{FORECAST_BASE}/forecasts/history/province_weekly_forecast_history.json")

province_weekly_df = pd.DataFrame(province_weekly_history)
province_weekly_df["week_end"] = pd.to_datetime(province_weekly_df["week_end"])
province_weekly_df["generated_date"] = pd.to_datetime(province_weekly_df["generated_date"])

active_province_weekly = province_weekly_df[province_weekly_df["week_end"] >= today]
active_province_weekly = active_province_weekly.sort_values("generated_date").drop_duplicates(subset=["week_start", "province"], keep="last")
active_province_weekly = active_province_weekly.sort_values(["week_start", "province"])

active_province_weekly_records = active_province_weekly.to_dict(orient="records")
for r in active_province_weekly_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/province_weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_province_weekly_records, indent=2), overwrite=True)
print(f"Active province weekly: {len(active_province_weekly_records)} records")

count = save_history_as_excel(blob_service, province_weekly_history, f"{FORECAST_BASE}/forecasts/history/province_weekly_forecast_history.xlsx")
print(f"Saved province weekly Excel: {count} rows")

Save province - monthly

In [0]:
province_monthly_records = monthly_with_geo.groupby(["province_name", "month_start"])["predicted_units"].sum().reset_index()
province_monthly_records["generated_date"] = today_str
province_monthly_records = province_monthly_records.rename(columns={"province_name": "province"}).to_dict(orient="records")

province_monthly_history = append_json_history(blob_service, province_monthly_records, f"{FORECAST_BASE}/forecasts/history/province_monthly_forecast_history.json")

province_monthly_df = pd.DataFrame(province_monthly_history)
province_monthly_df["month_start"] = pd.to_datetime(province_monthly_df["month_start"])
province_monthly_df["generated_date"] = pd.to_datetime(province_monthly_df["generated_date"])
province_monthly_df["month_end"] = province_monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_province_monthly = province_monthly_df[province_monthly_df["month_end"] >= today]
active_province_monthly = active_province_monthly.sort_values("generated_date").drop_duplicates(subset=["month_start", "province"], keep="last")
active_province_monthly = active_province_monthly.sort_values(["month_start", "province"])

active_province_monthly_records = active_province_monthly.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_province_monthly_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/province_monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_province_monthly_records, indent=2), overwrite=True)
print(f"Active province monthly: {len(active_province_monthly_records)} records")

count = save_history_as_excel(blob_service, province_monthly_history, f"{FORECAST_BASE}/forecasts/history/province_monthly_forecast_history.xlsx")
print(f"Saved province monthly Excel: {count} rows")